In [71]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent

raw_path = BASE_DIR / "data" / "raw"
processed_path = BASE_DIR / "data" / "processed"

print(raw_path)
print(processed_path)

c:\Users\ASUS\OneDrive\Desktop\bluestock-mf-capstone\data\raw
c:\Users\ASUS\OneDrive\Desktop\bluestock-mf-capstone\data\processed


In [72]:
files = [
    "01_fund_master.csv",
    "02_nav_history.csv",
    "03_aum_by_fund_house.csv",
    "04_monthly_sip_inflows.csv",
    "05_category_inflows.csv",
    "06_industry_folio_count.csv",
    "07_scheme_performance.csv",
    "08_investor_transactions.csv",
    "09_portfolio_holdings.csv",
    "10_benchmark_indices.csv"
]

files

['01_fund_master.csv',
 '02_nav_history.csv',
 '03_aum_by_fund_house.csv',
 '04_monthly_sip_inflows.csv',
 '05_category_inflows.csv',
 '06_industry_folio_count.csv',
 '07_scheme_performance.csv',
 '08_investor_transactions.csv',
 '09_portfolio_holdings.csv',
 '10_benchmark_indices.csv']

In [73]:
summary = []

for file in files:
    df = pd.read_csv(raw_path / file)

    summary.append({
        "File": file,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": df.isnull().sum().sum(),
        "Duplicates": df.duplicated().sum(),
        "Quality_factor":round((((df.shape[0]*df.shape[1])-(df.isnull().sum().sum()))/(df.shape[0]*df.shape[1]))*100)
    })

summary_df = pd.DataFrame(summary)

summary_df

,File,Rows,Columns,Missing Values,Duplicates,Quality_factor
0,01_fund_master.csv,40,15,0,0,100
1,02_nav_history.csv,46000,3,0,0,100
2,03_aum_by_fund_house.csv,90,5,0,0,100
3,04_monthly_sip_inflows.csv,48,6,12,0,96
4,05_category_inflows.csv,144,3,0,0,100
5,06_industry_folio_count.csv,21,6,0,0,100
6,07_scheme_performance.csv,40,19,0,0,100
7,08_investor_transactions.csv,32778,13,0,0,100
8,09_portfolio_holdings.csv,322,8,0,0,100
9,10_benchmark_indices.csv,8050,3,0,0,100


In [74]:
msi = pd.read_csv(raw_path / '04_monthly_sip_inflows.csv')

In [75]:
msi.isnull().sum()

month                         0
sip_inflow_crore              0
active_sip_accounts_crore     0
new_sip_accounts_lakh         0
sip_aum_lakh_crore            0
yoy_growth_pct               12
dtype: int64

The first 12 months do not have prior-year observations, so YoY growth cannot be computed. These nulls represent unavailable data rather than missing data and were retained to preserve data integrity, So I don't remove it by droping them

02_nav_history.csv

In [76]:
nav = pd.read_csv(raw_path/'02_nav_history.csv')
nav.shape

(46000, 3)

In [77]:
nav

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692
...,...,...,...
45995,149324,2026-05-25,292.4810
45996,149324,2026-05-26,291.2707
45997,149324,2026-05-27,288.8007
45998,149324,2026-05-28,280.6873


In [78]:
nav['date'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 46000 entries, 0 to 45999
Series name: date
Non-Null Count  Dtype 
--------------  ----- 
46000 non-null  object
dtypes: object(1)
memory usage: 359.5+ KB


In [79]:
nav['date'] = pd.to_datetime(nav['date'])

In [80]:
nav = nav.sort_values(by=['amfi_code','date'])
nav

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639
...,...,...,...
45995,149324,2026-05-25,292.4810
45996,149324,2026-05-26,291.2707
45997,149324,2026-05-27,288.8007
45998,149324,2026-05-28,280.6873


In [81]:
nav['nav'].isnull().sum()

np.int64(0)

 Ye har fund ke andar previous NAV se fill karega.

In [82]:
nav['nav'] = nav.groupby('amfi_code')['nav'].ffill()

In [83]:
invalid_nav = nav[nav['nav'] <= 0]

print("Invalid NAV Rows:", len(invalid_nav))

Invalid NAV Rows: 0


In [84]:
nav.to_csv(
    processed_path / "02_nav_history_clean.csv",
    index=False
)

print("02_nav_history_clean.csv saved successfully")

02_nav_history_clean.csv saved successfully


In [85]:
tx = pd.read_csv(raw_path / "08_investor_transactions.csv")

print(tx.shape)
tx.head()

(32778, 13)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [86]:
tx = tx.drop_duplicates()

tx['transaction_date'] = pd.to_datetime(
    tx['transaction_date']
)

tx['transaction_type'] = (
    tx['transaction_type']
    .str.strip()
    .str.title()
)

tx = tx[tx['amount_inr'] > 0]

tx.to_csv(
    processed_path/"clean_transactions.csv",
    index=False
)

In [87]:
fund = pd.read_csv(raw_path/"01_fund_master.csv")

fund = fund.drop_duplicates()

fund['launch_date'] = pd.to_datetime(
    fund['launch_date'],
    errors='coerce'
)

fund.to_csv(
    processed_path/"clean_fund_master.csv",
    index=False
)

In [89]:
perf = pd.read_csv(
    raw_path/"07_scheme_performance.csv"
)

numeric_cols = [
    'return_1yr_pct',
    'return_3yr_pct',
    'return_5yr_pct',
    'alpha',
    'beta',
    'sharpe_ratio'
]

for col in numeric_cols:
    perf[col] = pd.to_numeric(
        perf[col],
        errors='coerce'
    )

perf.to_csv(
    processed_path/"clean_performance.csv",
    index=False
)